<a href="https://colab.research.google.com/github/essanchristian-maker/DI-Bootcamp/blob/master/Week9_Day3_dailychallenge_de_agentic_agent_student_DC.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Tiny Agent with Tools ?

All open source: tiny local model, wiki library, no API keys. Run top-to-bottom.

In [ ]:
!pip install -q smolagents[transformers] wikipedia

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 161.5/161.5 kB 5.2 MB/s eta 0:00:00


## 1) Define KB

In [ ]:
#To-Do: you can add your own knowledge base snippets here
kb_snippets = [
    {'source': 'kb:agentic', 'text': 'Agentic AI loops plan, choose tools, and reflect before answering.'},
    {'source': 'kb:tools', 'text': 'Useful tools: math, search, and domain-specific lookup.'},
    {'source': 'kb:citation', 'text': 'Always cite where evidence came from to stay transparent.'},
    {'source': 'kb:brevity', 'text': 'Keep answers concise (2-4 sentences).'},
    {'source': 'kb:followup', 'text': 'If evidence is missing, say so and propose a follow-up question.'},
]
print('KB entries:', len(kb_snippets))


KB entries: 5


## 2) Define tools

In [ ]:
from smolagents import Tool, TransformersModel, ToolCallingAgent

# Defining the KB snippets here to ensure they are available to the tool
kb_snippets = [
    {'source': 'kb:agentic', 'text': 'Agentic AI loops plan, choose tools, and reflect before answering.'},
    {'source': 'kb:tools', 'text': 'Useful tools: math, search, and domain-specific lookup.'},
    {'source': 'kb:citation', 'text': 'Always cite where evidence came from to stay transparent.'},
    {'source': 'kb:brevity', 'text': 'Keep answers concise (2-4 sentences).'},
    {'source': 'kb:followup', 'text': 'If evidence is missing, say so and propose a follow-up question.'},
]

class KBLookupTool(Tool):
    name = "kb_lookup_tool"
    description = "Looks up relevant information from a custom knowledge base."
    inputs = {"query": {"type": "string", "description": "The search query to match against the KB."}}
    output_type = "string"

    def __init__(self, kb):
        super().__init__()
        self.kb = kb

    def forward(self, query: str) -> str:
        q = query.lower()
        matches = [
            f"[{item['source']}] {item['text']}"
            for item in self.kb
            if any(w in item["text"].lower() for w in q.split())
        ]
        return " ".join(matches) if matches else "No KB match."

class MathTool(Tool):
    name = "math_tool"
    description = "Add or multiply two numbers."
    inputs = {
        "a": {"type": "number", "description": "First number"},
        "b": {"type": "number", "description": "Second number"},
        "op": {"type": "string", "description": "Operation: 'add' or 'multiply'", "nullable": True}
    }
    output_type = "string"

    def forward(self, a: float, b: float, op: str = "add") -> str:
        if op == "multiply":
            return str(a * b)
        return str(a + b)

kb_tool = KBLookupTool(kb_snippets)
math_tool = MathTool()

## 3) Model (tiny local)

In [ ]:
import torch
MODEL_ID = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

# Suppression de l'argument dtype explicite qui cause l'erreur de génération
model = TransformersModel(
    model_id=MODEL_ID,
    device_map="auto",
    max_new_tokens=200,
    torch_dtype=torch.float16
)

print("Model ready:", MODEL_ID)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Model ready: TinyLlama/TinyLlama-1.1B-Chat-v1.0


## 4) Agent

In [ ]:
agent = ToolCallingAgent(
    tools=[kb_tool, math_tool],
    model=model,
    max_steps=5,
    instructions=(
        "You are an agentic AI that uses tools to answer questions. "
        "Keep answers concise (2-4 sentences). Always cite sources like [kb:source] if using the KB. "
        "If no evidence is found, say so and propose a follow-up."
    ),
)

print("Agent initialized.")

Agent initialized.


## 5) Test queries

In [ ]:
tests = [
    "Add 12 and 30.",
    "Multiply 7 by 6.",
    "What is an agentic AI loop?",
]

for q in tests:
    print(f"\n{'='*20}\nRunning Query: {q}\n{'='*20}")
    try:
        # Directly printing the run to see steps
        final_answer = agent.run(q)
        print(f"\nRESULT: {final_answer}")
    except Exception as e:
        print(f"Error processing '{q}': {e}")


Running Query: Add 12 and 30.


╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ Add 12 and 30.                                                                                                  │
│                                                                                                                 │
╰─ TransformersModel - TinyLlama/TinyLlama-1.1B-Chat-v1.0 ────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Error while generating output:
The following `model_kwargs` are not used by the model: ['dtype'] (note: typos in the generate arguments will also 
show up in this list)

[Step 1: Duration 0.01 seconds]

...

╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ What is an agentic AI loop?                                                                                     │
│                                                                                                                 │
╰─ TransformersModel - TinyLlama/TinyLlama-1.1B-Chat-v1.0 ────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Error while generating output:
The following `model_kwargs` are not used by the model: ['dtype'] (note: typos in the generate arguments will also 
show up in this list)

[Step 1: Duration 0.01 seconds]

Error processing 'What is an agentic AI loop?': Error while generating output:
The following `model_kwargs` are not used by the model: ['dtype'] (note: typos in the generate arguments will also show up in this list)
